In [1]:
import numpy as np
np.random.seed(1337) 
import pandas as pd
import keras
import keras.regularizers
import glob
import os
import itertools
import seaborn as sns


from sklearn.preprocessing import MinMaxScaler, StandardScaler, MaxAbsScaler, Normalizer
from sklearn.metrics import confusion_matrix, recall_score, f1_score, precision_score

from keras_self_attention import SeqSelfAttention
from keras.utils import custom_object_scope

import matplotlib.pyplot as plt

# Import path configuration
import sys
sys.path.insert(0, '..')  # Add parent directory to path
from config import setup_workdir, get_path

2026-03-09 15:40:54.447867: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-09 15:40:54.670255: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-09 15:40:54.675388: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-03-09 15:40:54.675411: I tensorflow/compiler/xla

In [8]:
# Find the best model from cross-validation folds
# Using the no-attention model which showed best interpretability and clean saliency maps

# Choose which model architecture to use:
model_type = 'no_attention'  # Options: 'csd' (with attention), 'no_attention', 'minimal'

if model_type == 'csd':
    # Original model with attention mechanism
    folds_dir = get_path('csd_models') / 'cv_folds'
    fold_pattern = 'csd_fold_*_model.h5'
    use_attention = True
elif model_type == 'no_attention':
    # No-attention model (best interpretability)
    folds_dir = get_path('csd_models').parent / 'csd_no_attention' / 'cv_folds'
    fold_pattern = 'csd_fold_*_model.h5'
    use_attention = False
elif model_type == 'minimal':
    # Minimal architecture model
    folds_dir = get_path('csd_models').parent / 'csd_minimal' / 'cv_folds'
    fold_pattern = 'minimal_fold_*_model.h5'
    use_attention = False
else:
    raise ValueError(f"Unknown model_type: {model_type}")

fold_models = sorted(folds_dir.glob(fold_pattern))

if not fold_models:
    raise FileNotFoundError(f"No fold models found in {folds_dir}")

print(f"Loading {model_type} model")
print(f"Found {len(fold_models)} fold models")

# Load the first fold model (or specify best_fold if known)
#best_model_path = fold_models[0]  # Default to first fold

# If you know the best fold from training notebook 4, you can set it here:
best_fold = 9  # Example: if fold 3 was best
best_model_path = folds_dir / f'csd_fold_{best_fold}_model.h5'

print(f"Loading model: {best_model_path.name}")

if use_attention:
    with custom_object_scope({'SeqSelfAttention': SeqSelfAttention}):
        load_model = keras.models.load_model(best_model_path)
else:
    load_model = keras.models.load_model(best_model_path)

# Set working directory to processed experimental PDFs
setup_workdir('experimental_processed')
files = glob.glob('*processed.gr')

Loading no_attention model
Found 10 fold models
Loading model: csd_fold_9_model.h5
Working directory: /workspace/home/pdf-nn-data/experimental_pdfs/processed


In [9]:
# Load and preprocess experimental PDF data
raw_data_points = []
filenames = []

for f in files:
    df = pd.read_csv(f, usecols=[1], skiprows=1, header=None, delim_whitespace=True, skipfooter=1, engine='python')
    raw_data_points.append(df.values.ravel())
    filenames.append(os.path.basename(f))

raw_data_points = np.array(raw_data_points)

# Normalize the data (same preprocessing as training)
normalize = Normalizer()
data_points = normalize.fit_transform(raw_data_points)

print(f"Loaded {len(data_points)} PDF files for prediction")

Loaded 27 PDF files for prediction


In [10]:
# Run predictions
y_pred_prob = load_model.predict(data_points)

# Get top 2 predictions for each sample
top2_indices = np.argsort(y_pred_prob, axis=1)[:, -2:][:, ::-1]  # Top 2, descending
top2_probs = np.take_along_axis(y_pred_prob, top2_indices, axis=1)

# Map predictions to nuclearity labels
# Model trained with labels 1-9 (nuclearity) and 10 (polymer)
def pred_to_nuclearity(pred):
    if pred == 10:
        return "polymer"
    else:
        return str(pred)

def clip_filename(fname, max_len=32):
    """Truncate filename if too long, adding ellipsis."""
    if len(fname) > max_len:
        return fname[:max_len-3] + "..."
    return fname

# Display results
print("=" * 85)
print("PREDICTION RESULTS")
print("=" * 85)
print(f"\n{'Filename':<32} {'1st Prediction':>14} {'Conf.':>8} {'2nd Prediction':>14} {'Conf.':>8}")
print("-" * 85)

results = []
for i, fname in enumerate(filenames):
    pred1 = top2_indices[i, 0]
    pred2 = top2_indices[i, 1]
    conf1 = top2_probs[i, 0]
    conf2 = top2_probs[i, 1]
    
    nuc1 = pred_to_nuclearity(pred1)
    nuc2 = pred_to_nuclearity(pred2)
    
    display_name = clip_filename(fname)
    print(f"{display_name:<32} {nuc1:>14} {conf1:>7.1%} {nuc2:>14} {conf2:>7.1%}")
    results.append({
        'filename': fname,  # Keep full filename in CSV
        'prediction_1': nuc1, 
        'confidence_1': conf1,
        'prediction_2': nuc2,
        'confidence_2': conf2
    })

# Save predictions to CSV
results_df = pd.DataFrame(results)
output_path = get_path('labels') / 'predictions.csv'
results_df.to_csv(output_path, index=False)
print(f"\nPredictions saved to: {output_path}")

1/1 [==============================] - 0s 105ms/step
PREDICTION RESULTS

Filename                         1st Prediction    Conf. 2nd Prediction    Conf.
-------------------------------------------------------------------------------------
DF184_5reps_acc_01_processed.gr         polymer   97.4%              8    1.4%
DF211_15reps_acc_ext01_proces...              4   63.4%              2   12.6%
Gd_tfa_iPrOH_kapton15_process...              4   97.0%              8    2.9%
DF211_processed.gr                            4   63.4%              2   12.6%
pdf42_BM39_Tb4tfa_kapton05_pr...              4   59.0%              7   40.9%
DF221_01_processed.gr                         4   82.5%              6    6.6%
DF471_0.7mm_boro_single_full_...        polymer   97.0%              2    1.9%
DF210dr_5reps_acc_01_processe...              4   93.7%              6    2.1%
DF186_5reps_processed.gr                polymer   73.1%              4   11.6%
KM114r_acc_15reps_1Kapton_pro...              4  